# Higher-order local fields and variable operators

This notebook solves small native examples and inspects the complete six-point
series archived in `examples/results/native-extensions`. The fields used as
references are explicit analytical expressions; these are not external solver
comparisons. Every spatial image shows the actual macrotriangulation. Profiles
preserve one-sided local values at macro interfaces.

The experiments cover primal Darcy P1–P4, Taylor–Hood and equal-order USFEM with
variable tensor resistance, conservative variable-coefficient RAD, Galerkin/SUPG
outflow layers, and P2/P3 transient heat. The gallery generator records all local
and trace degrees, subdivisions, quadrature orders, errors and field hashes.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
from IPython.display import SVG, display
from threadpoolctl import threadpool_limits

root = Path.cwd()
if not (root / "examples").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "examples"))
from manufactured import darcy_pressure, darcy_source, darcy_flux, stokes_velocity
from native_extension_data import (
    BoundaryLayer, rad_diffusion, rad_velocity, rad_reaction, rad_source,
    sine, heat_exact, heat_source, flow_source, resistance,
)
from pymhm import TriangleMesh, FaceSpace, SkeletonSpace, solve_darcy, solve_brinkman, solve_transport, solve_heat

result_path = root / "examples/results/native-extensions"
report = json.loads((result_path / "report.json").read_text())
expected_counts = {"darcy": 24, "flow": 12, "rad": 12, "layer": 24, "heat": 12}
for family, count in expected_counts.items():
    assert len(report[family]) == count
    for row in report[family]:
        if "fields" in row:
            field = row["fields"]
            assert hashlib.sha256((result_path / field["file"]).read_bytes()).hexdigest() == field["sha256"]
print({family: len(report[family]) for family in expected_counts})


## P3 primal Darcy: pressure and its raw physical gradient

The pressure is `cos(pi*x)*cos(pi*y)`, the permeability is one, and the source is
`2*pi**2*p`. The full boundary carries its nonhomogeneous Dirichlet values.
P3 local pressure uses P2 skeletal moments. `flux_l2_error` evaluates
`-grad(p_h)`; this raw primal flux is not an H(div)-conforming reconstruction.


In [ ]:
mesh = TriangleMesh.unit_square(1)
skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(2) for _ in mesh.faces))
with threadpool_limits(limits=1):
    darcy = solve_darcy(
        mesh, degree=3, skeleton=skeleton, local_refinement=2,
        source=darcy_source, dirichlet=darcy_pressure, quadrature_order=8,
    )
errors = {"pressure_l2": darcy.l2_error(darcy_pressure, 10),
          "raw_flux_l2": darcy.flux_l2_error(darcy_flux, 10),
          "macro_balance": float(np.max(np.abs(darcy.conservation_residuals(10))))}
assert errors["macro_balance"] < 1e-11
errors


In [ ]:
display(SVG(filename=str(root / "docs/figures/high-order/darcy-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/darcy-p4.svg")))

display(SVG(filename=str(root / "docs/figures/high-order/darcy-p4-flux.svg")))


## Variable tensor Brinkman resistance

The positive definite resistance is
`Gamma=(1+x+2*y)*array([[20,3],[3,1]])`. The force is the independently
differentiated Stokes force plus `Gamma @ u_exact`. Both components are coupled;
USFEM includes the complete tensor resistance, velocity Laplacian and pressure
gradient in its residual stabilization.

The six-point series holds 32 macrotriangles fixed, uses vector P1 traces and
`2*s` local subdivisions for TH P2/P1 and USFEM P2/P2. The third map panel is
`norm(u_h-u_exact)`, not a difference of magnitudes.


In [ ]:
flow_trace = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces), 2)
with threadpool_limits(limits=1):
    brinkman = solve_brinkman(
        mesh, degree=2, formulation="usfem", drag=resistance, source=flow_source,
        skeleton=flow_trace, local_refinement=2, quadrature_order=8,
    )
print("Small USFEM P2 velocity error:", brinkman.l2_error(stokes_velocity, 8))
for method in ("taylor-hood", "usfem"):
    rows = [row for row in report["flow"] if row["method"] == method]
    print(method, [(row["segments"], row["velocity_l2"], row["pressure_l2"]) for row in rows])
display(SVG(filename=str(root / "docs/figures/high-order/flow-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/tensor-brinkman.svg")))

display(SVG(filename=str(root / "docs/figures/high-order/tensor-brinkman-pressure.svg")))


## Variable conservative RAD and an independent small solve

The equation is `-div(K grad(u)) + div(beta*u) + c*u = f`, with
`K=.05*(1+x+y)*[[2,.3],[.3,1]]`, `beta=(1+x,.5+y)` and `c=.5+x*y`.
The analytic data supply the full conservative forcing for `sin(pi*x)*sin(pi*y)`.
The coefficient derivatives are explicit: `div(K)=(.115,.065)` and `div(beta)=2`.
They enter the full SUPG residual along with the polynomial Hessian.

The skeletal multiplier has the Robin convention
`(-K @ grad(u) + beta*u/2) @ n`; it is not the full conservative flux.


In [ ]:
scalar_trace = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))
with threadpool_limits(limits=1):
    rad = solve_transport(
        mesh, degree=2, skeleton=scalar_trace, local_refinement=2,
        diffusion=rad_diffusion, diffusion_divergence=(0.115, 0.065),
        velocity=rad_velocity, velocity_divergence=2.0, reaction=rad_reaction,
        source=rad_source, stabilization="supg", quadrature_order=8,
    )
print("SUPG P2 L2 error:", rad.l2_error(sine, 10))
display(SVG(filename=str(root / "docs/figures/high-order/rad-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/variable-rad.svg")))


## Outflow layers: distinguish error reduction from positivity

For `epsilon=.02` or `.005`, the exact field is
`x-(exp((x-1)/epsilon)-exp(-1/epsilon))/(1-exp(-1/epsilon))`.
It solves `-epsilon*Delta(u)+u_x=1`. Only the vertical edges have zero values;
the horizontal edges require the full nonzero exact boundary function.

The recorded Galerkin and SUPG P1 cases use 32 macrotriangles, P1 traces,
`4*s` local subdivisions, assembly order 20 and error order 24. Extrema over all
local nodal coefficients are the exact extrema of the broken P1 field. They
include weak boundary deviations. SUPG does not enforce a discrete maximum
principle. Vertical profile markers show every actual macroface crossing.


In [ ]:
layer_mesh = TriangleMesh.unit_square(2)
layer_trace = SkeletonSpace(layer_mesh, tuple(FaceSpace.uniform(1) for _ in layer_mesh.faces))
layer_exact = BoundaryLayer(0.005)
with threadpool_limits(limits=1):
    layer = solve_transport(
        layer_mesh, diffusion=0.005, velocity=(1.0, 0.0), source=1.0,
        dirichlet=layer_exact.value, degree=1, skeleton=layer_trace,
        local_refinement=4, stabilization="supg", quadrature_order=20,
    )
print("Small SUPG layer error:", layer.l2_error(layer_exact.value, 24))
for epsilon in (0.02, 0.005):
    exact = BoundaryLayer(epsilon)
    for method in ("galerkin", "supg"):
        rows = [row for row in report["layer"]
                if row["epsilon"] == epsilon and row["stabilization"] == method]
        print(epsilon, method,
              [(row["segments"], row["scalar_l2"], row["undershoot"], row["overshoot"])
               for row in rows])
    print("Exact maximum:", exact.maximum)
display(SVG(filename=str(root / "docs/figures/high-order/layer-errors.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/layer-profiles.svg")))


## P2/P3 heat with a time-linear exact field

Set `u_exact=(1+t)*sin(pi*x)*sin(pi*y)` and
`f=(1+2*pi**2*(1+t))*sin(pi*x)*sin(pi*y)`. Its backward-Euler difference quotient
is exact in time, so the four-step archived curves primarily assess spatial
approximation and initial interpolation. They do not claim higher temporal order.


In [ ]:
with threadpool_limits(limits=1):
    heat = solve_heat(
        mesh, [0.0, 0.05, 0.1], degree=2, skeleton=scalar_trace,
        local_refinement=2, initial=sine, source=heat_source, quadrature_order=8,
    )[-1]
print("Heat P2 error at t=.1:", heat.l2_error(lambda points: heat_exact(points, .1), 10))
display(SVG(filename=str(root / "docs/figures/high-order/heat-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/heat-p3.svg")))


## Recompute the full gallery

Run `pixi run -e notebooks gallery-native-extensions --workers 4` from the
repository. The optional `--sections` argument selects families; all completed
cases are checkpointed. `--reuse-results` redraws the archived data without PDE
solves. The numerical report records separate source hashes for each family and
hashes for every displayed field archive.
